> ⚠️ **This notebook trains a deep-learning model and is GPU-recommended.**
> The outputs below are NOT pre-rendered — execute the notebook on a CUDA
> / MPS-equipped host to populate them. The code path was validated end
> to end in the [`omicverse#797` integration test suite][1] (`scvi-tools`
> 1.3.0, with the same kwarg-split router used here), so it is known to
> run; only the rendering depends on you executing it.
>
> Quick recipe:
> ```bash
> jupyter nbconvert --to notebook --execute --inplace \
>     docs/Tutorials-single/batch/zoo/<this-notebook>.ipynb
> ```
> on a node where `torch.cuda.is_available()` is `True`.
>
> [1]: https://github.com/omicverse/omicverse/pull/797


# Batch correction with scANVI (semi-supervised)

scANVI (Xu et al., *Mol Syst Biol* 2021) extends scVI with a supervised classifier head trained on partially-labelled cells. Cells without a confident label get a placeholder (here `'Unknown'`); scANVI both batch-corrects and predicts labels for the unknowns. The corrected latent lands in `adata.obsm['X_scANVI']`, predicted labels in `adata.obs['scANVI_predicted_labels']`.

This is one of the **omicverse batch-correction zoo** tutorials. See [batch/index](../index.md) for the overview / decision tree, or [../t_single_batch](../t_single_batch.ipynb) for the side-by-side comparison of every backend on a real benchmark.

## Load a 2-batch demo from pbmc3k

We use the canonical 10x pbmc3k dataset and synthesise a 2-batch label by random assignment, then plant a gene-shift on `batch_B` so the uncorrected UMAP shows a visible batch effect. This keeps the notebook self-contained and fast (~2 min end-to-end) — for a real multi-donor benchmark with [scib-metrics] scoring, see [../t_single_batch](../t_single_batch.ipynb).

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np
import pandas as pd

# NeurIPS 2021 multimodal hematopoiesis dataset — 3 real donor batches
# (s1d3, s2d1, s3d7), pre-annotated `cell_type` and raw `layers['counts']`.
# Same datasets used by the overview notebook ../t_single_batch.ipynb.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.obs['batch'] = adata.obs['batch'].astype('category')

# Subsample to a quick-running ~6 000 cells × 3 batches so the CPU
# backends (harmony / combat / scanorama / cca) finish in well under a
# minute. Drop this line for a full-resolution run.
sc.pp.subsample(adata, n_obs=6000, random_state=0)
adata

## Preprocess + PCA + cluster

Same QC → HVG-pearson → log-norm → PCA pipeline shared across every backend in the zoo. A quick Leiden cluster gives a synthetic `celltype` label that scANVI / scPoli can use as a prototype anchor.

In [ ]:
# Standard omicverse preprocess (QC → HVG-via-pearson → log-norm → PCA).
# QC thresholds are loose because the NeurIPS data is already filtered.
adata = ov.pp.qc(adata, tresh={'mito_perc': 0.2, 'nUMIs': 200,
                                 'detected_genes': 100})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson', n_HVGs=2000,
                         batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features].copy()
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=30)

# Neighbours graph for the pre-correction UMAP.
sc.pp.neighbors(adata, use_rep='scaled|original|X_pca', n_neighbors=15)

# The NeurIPS adata already carries a real `cell_type` annotation —
# rename it to `celltype` for the wrapper's expected schema. No Leiden
# needed; the labels are pre-annotated by the dataset authors.
adata.obs['celltype'] = adata.obs['cell_type'].astype('category')
adata

## Uncorrected baseline

The planted batch effect is visible in the uncorrected UMAP:

In [ ]:
# Pre-correction UMAP shows the planted batch effect.
sc.tl.umap(adata, min_dist=0.3)
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
ov.pl.embedding(adata, basis='X_umap_uncorrected',
                color=['batch', 'celltype'],
                frameon='small', wspace=0.5)

## Run `ov.single.batch_correction(methods='scanvi')`

For the scvi-tools family backends, the wrapper auto-routes `**kwargs` between the model's `__init__` (architecture) and `.train()` (optimisation) destinations. See the **Key parameters** section below.

In [ ]:
# Mark every 5th cell as 'Unknown' so scANVI has something
# to predict. In real workflows the labels come from prior
# annotation; the 'Unknown' value is for cells you do NOT
# yet have confident labels for.
labels = adata.obs['celltype'].astype(object)
labels.iloc[::5] = 'Unknown'
adata.obs['celltype_for_scanvi'] = labels.astype('category')

model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='scANVI',
    # Required by scANVI:
    labels_key='celltype_for_scanvi',
    unlabeled_category='Unknown',
    # Architecture:
    n_hidden=64, n_latent=10,
    # Optimisation — scvi-tools auto-picks CUDA / MPS / CPU.
    max_epochs=10,
)
model

## Corrected embedding

Every backend writes its corrected representation to a stable obsm key — for this one it is `adata.obsm['X_scANVI']`. We project via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_scanvi'] = ov.utils.mde(adata.obsm['X_scANVI'])
ov.pl.embedding(
    adata,
    basis='X_mde_scanvi',
    color=['batch', 'celltype', 'scANVI_predicted_labels'],
    frameon='small',
    wspace=0.5,
)

## Key parameters

**Required**:
- `labels_key` — obs column with cell-type labels.
- `unlabeled_category` — value meaning "predict this" (default `'Unknown'`).

**Architecture / optimisation**: same as scVI — the kwarg splitter routes `n_latent` / `max_epochs` / etc. to the correct side.

**Outputs**:
- `adata.obsm['X_scANVI']` — corrected latent.
- `adata.obs['scANVI_predicted_labels']` — predicted labels.


## Related tutorials

- [t_batch_scvi](t_batch_scvi.ipynb) — unsupervised when you have no labels.
- [t_batch_scpoli](t_batch_scpoli.ipynb) — alternative semi-supervised approach via per-condition prototypes.

For the full side-by-side comparison with scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).